In [31]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [7]:
from connexion import Connexion as cnn
fa = """SELECT * FROM public."Fact_Activity";"""
dc= """SELECT * FROM public."Dim_Customer";"""
dpp= """SELECT * FROM public."Dim_Products";"""

dim_customer = cnn.connect_to_dw(dc)
dim_customer = pd.DataFrame(dim_customer)
dim_products = cnn.connect_to_dw(dpp)
dim_products = pd.DataFrame(dim_products)
fact_activity = cnn.connect_to_dw(fa)
fact_activity = pd.DataFrame(fact_activity)


In [15]:
# Merge the customer data with activity data
activity_with_customer = pd.merge(fact_activity, dim_customer, left_on='Customer_FK', right_on='Customer_PK', how='left')

# Merge the result with products data
full_data = pd.merge(activity_with_customer, dim_products, left_on='Products_FK', right_on='Products_PK', how='left')

# Select only necessary columns: Customer ID, Product Name, and Customer Reviews
user_item_data = full_data[['id_x', 'name', 'Customer_Reviews']].copy()

# Rename columns for clarity
user_item_data.columns = ['Customer_ID', 'Game_Name', 'Review_Score']

# Create a pivot table to form the user-item matrix
user_item_matrix = user_item_data.pivot_table(index='Customer_ID', columns='Game_Name', values='Review_Score')

# Display the structure of the resulting user-item matrix
user_item_matrix.head(), user_item_matrix.shape


(Game_Name    Autobahn  Bamboozler  Beagle Brigade Airfield  Boomerang  \
 Customer_ID                                                             
 1            4.300000    3.750000                 4.133333   3.900000   
 2            4.140000    4.075000                 3.866667   3.980000   
 3            4.000000    4.240000                 4.016667   4.214286   
 4            4.583333    4.433333                 4.200000   4.360000   
 5            4.250000    4.180000                 4.240000   4.257143   
 
 Game_Name    Camp Bus  Caribbean Cooler  Charlie Brown's Windup  Coconut Cove  \
 Customer_ID                                                                     
 1            4.650000          4.375000                4.325000      3.971429   
 2            4.200000          4.385714                4.114286      4.166667   
 3            4.140000          4.266667                4.462500      4.360000   
 4            4.357143          4.088889                4.125000      

In [18]:
# Convert the user-item matrix to a DataFrame with explicit rows for missing values
ratings_df = user_item_matrix.stack().reset_index()
ratings_df.columns = ['Customer_ID', 'Game_Name', 'Review_Score']

# Define a custom dataset class
class ReviewDataset(Dataset):
    def __init__(self, users, games, ratings):
        self.users = torch.tensor(users, dtype=torch.long)
        self.games = torch.tensor(games, dtype=torch.long)
        self.ratings = torch.tensor(ratings, dtype=torch.float)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return {
            'user': self.users[idx],
            'game': self.games[idx],
            'rating': self.ratings[idx]
        }


# Encoding user IDs and game names to continuous indices for embedding layers
user_ids = ratings_df['Customer_ID'].astype('category').cat.codes.values
game_names = ratings_df['Game_Name'].astype('category').cat.codes.values
ratings = ratings_df['Review_Score'].values

# Splitting the data
train_indices, test_indices = train_test_split(np.arange(len(ratings)), test_size=0.2, random_state=42)
train_data = ReviewDataset(user_ids[train_indices], game_names[train_indices], ratings[train_indices])
test_data = ReviewDataset(user_ids[test_indices], game_names[test_indices], ratings[test_indices])

# Data loaders
train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

# Checking a sample from the dataset
next(iter(train_loader))


{'user': tensor([73, 21, 68, 55,  9, 88, 77, 19, 78, 34]),
 'game': tensor([12, 29, 19, 23, 28,  7, 17, 31,  3,  0]),
 'rating': tensor([3.8400, 4.0667, 4.2500, 4.2286, 4.1286, 4.2500, 4.1857, 4.3667, 4.2857,
         4.0333])}

In [35]:
# Define the model
class CollaborativeFiltering(nn.Module):
    def __init__(self, num_users, num_games, embedding_size=50):
        super(CollaborativeFiltering, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.game_embedding = nn.Embedding(num_games, embedding_size)
        self.fc1 = nn.Linear(embedding_size * 2, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, user, game):
        user_embedded = self.user_embedding(user)
        game_embedded = self.game_embedding(game)
        concatenated = torch.cat([user_embedded, game_embedded], dim=1)
        out = F.relu(self.fc1(concatenated))
        out = self.fc2(out)
        return out.view(-1)

# Convert user_ids and game_names to PyTorch tensors
user_ids_tensor = torch.tensor(user_ids)
game_names_tensor = torch.tensor(game_names)

# Initialize the model
model = CollaborativeFiltering(num_users=len(torch.unique(user_ids_tensor)),
                               num_games=len(torch.unique(game_names_tensor)))


# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        users, games, ratings = batch['user'], batch['game'], batch['rating']
        optimizer.zero_grad()
        outputs = model(users, games)
        loss = criterion(outputs, ratings.float())
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(users)
    train_loss /= len(train_data)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}")

# Evaluation loop
model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch in test_loader:
        users, games, ratings = batch['user'], batch['game'], batch['rating']
        outputs = model(users, games)
        loss = criterion(outputs, ratings.float())
        test_loss += loss.item() * len(users)
test_loss /= len(test_data)
print(f"Test Loss: {test_loss:.4f}")


Epoch 1/500, Train Loss: 0.7535
Epoch 2/500, Train Loss: 0.1457
Epoch 3/500, Train Loss: 0.1309
Epoch 4/500, Train Loss: 0.1218
Epoch 5/500, Train Loss: 0.0984
Epoch 6/500, Train Loss: 0.1045
Epoch 7/500, Train Loss: 0.0736
Epoch 8/500, Train Loss: 0.0872
Epoch 9/500, Train Loss: 0.0618
Epoch 10/500, Train Loss: 0.0657
Epoch 11/500, Train Loss: 0.0637
Epoch 12/500, Train Loss: 0.0675
Epoch 13/500, Train Loss: 0.0585
Epoch 14/500, Train Loss: 0.0541
Epoch 15/500, Train Loss: 0.0532
Epoch 16/500, Train Loss: 0.0456
Epoch 17/500, Train Loss: 0.0439
Epoch 18/500, Train Loss: 0.0443
Epoch 19/500, Train Loss: 0.0466
Epoch 20/500, Train Loss: 0.0443
Epoch 21/500, Train Loss: 0.0403
Epoch 22/500, Train Loss: 0.0413
Epoch 23/500, Train Loss: 0.0377
Epoch 24/500, Train Loss: 0.0390
Epoch 25/500, Train Loss: 0.0394
Epoch 26/500, Train Loss: 0.0376
Epoch 27/500, Train Loss: 0.0365
Epoch 28/500, Train Loss: 0.0364
Epoch 29/500, Train Loss: 0.0397
Epoch 30/500, Train Loss: 0.0380
Epoch 31/500, Train

In [67]:
# Initialize lists to store predicted and actual ratings
predicted_ratings = []
actual_ratings = []

# Make predictions on the test dataset
model.eval()
with torch.no_grad():
    for batch in test_loader:
        users, games, ratings = batch['user'], batch['game'], batch['rating']
        outputs = model(users, games)
        predicted_ratings.extend(outputs.tolist())
        actual_ratings.extend(ratings.tolist())

# Calculate Mean Squared Error (MSE)
mse = np.mean((np.array(actual_ratings) - np.array(predicted_ratings))**2)

# Optionally, calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")


Mean Squared Error (MSE): 0.0414
